# Figure 5: model-derived vs human-interpreted uncertainty (two-panel redesign)

Regenerates the manuscript Figure 5. Panel A is the cumulative distribution functions of model standard deviation (across five base learners) and interpreter standard deviation (across three interpreters). Panel B is the joint scatter of the two, coloured by sampling stratum, with a 1:1 line and the Spearman rank correlation annotated. The data extraction is adapted from the original analysis so this notebook is self-contained; the two-panel visualisation is new.

Method note: the five base-learner probabilities are read at the centroid pixel at the native 4.77 m PlanetScope resolution. The interpreter standard deviation is computed over a 10 m buffer across the three interpreters.

## Inputs (Google Earth Engine)
- Six interpreter FeatureCollections: `projects/ee-islamkm/assets/interpreter{1,2,3}_200pts` and `..._200pts_certain`.
- Five base-learner probability images: `projects/ee-islamkm/assets/baselearner_{knn,logreg,rf,svc,xgb}_*` (the asset suffix is `_mngrv` for knn, logreg, rf and `_mgrv` for svc, xgb).

Requires a one-time `earthengine authenticate` and read access to that project.

## Outputs (in `outputs/`)
- `figure5_redesigned.png` (300 dpi) and `.pdf`.
- `reference_points_with_stats.csv`: the per-point DataFrame (interpreter and model standard deviations, stratum). Shipped so the figure can be reproduced without GEE.
- `spearman_correlation.txt`: overall and per-stratum rho and p-values.

## Reproducing the figure without GEE
Load `outputs/reference_points_with_stats.csv` into a DataFrame named `df`, then run section 6 (Spearman) and section 7 (figure). No GEE access is needed for that path.

## 1. Setup

In [ ]:
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

ee.Initialize()  # requires a one-time: earthengine authenticate

# Results (figure, per-point CSV, correlation text) are written here, next to this notebook.
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

COLOR_CONSENSUS = '#1f77b4'
COLOR_DISAGREE  = '#ff7f0e'

def ee_fc_to_df(fc):
    info = fc.getInfo()
    rows = [feat.get('properties', {}) for feat in info.get('features', [])]
    return pd.DataFrame(rows)

## 2. Helper functions

In [ ]:
def explode_multi_points(fc):
    def explode_feature(feature):
        geom = feature.geometry()
        geom_type = geom.type()
        is_container = ee.List(['MultiPoint', 'GeometryCollection']).contains(geom_type)
        def container():
            geoms = ee.List(geom.geometries())
            return ee.FeatureCollection(geoms.map(lambda g: ee.Feature(ee.Geometry(g)).copyProperties(feature)))
        def not_container():
            return ee.FeatureCollection([feature])
        return ee.Algorithms.If(is_container, container(), not_container())
    return ee.FeatureCollection(fc.map(explode_feature).flatten())

# Sample the first image band at a point's native pixel (the centroid value),
# rather than averaging over the 10 m buffer. The base-learner rasters are about
# 4.77 m native, so a 10 m buffer would otherwise blur two to three pixels.
def sample_image_at_point(img, geom, scale):
    band = ee.String(img.bandNames().get(0))
    d = img.reduceRegion(reducer=ee.Reducer.first(), geometry=geom, scale=scale)
    return ee.Number(d.get(band))

## 3. Load assets and tag by stratum

In [ ]:
interpreters = {
    1: ('projects/ee-islamkm/assets/interpreter1_200pts_certain', 'projects/ee-islamkm/assets/interpreter1_200pts'),
    2: ('projects/ee-islamkm/assets/interpreter2_200pts_certain', 'projects/ee-islamkm/assets/interpreter2_200pts'),
    3: ('projects/ee-islamkm/assets/interpreter3_200pts_certain', 'projects/ee-islamkm/assets/interpreter3_200pts'),
}

fc1, fc2, fc3 = [
    explode_multi_points(ee.FeatureCollection(cert)).map(lambda f: f.set('source', 'cert')).merge(
        ee.FeatureCollection(conf).map(lambda f: f.set('source', 'conf'))
    )
    for cert, conf in interpreters.values()
]

knn    = ee.Image('projects/ee-islamkm/assets/baselearner_knn_mngrv')
logreg = ee.Image('projects/ee-islamkm/assets/baselearner_logreg_mngrv')
rf     = ee.Image('projects/ee-islamkm/assets/baselearner_rf_mngrv')
svc    = ee.Image('projects/ee-islamkm/assets/baselearner_svc_mgrv')
xgb    = ee.Image('projects/ee-islamkm/assets/baselearner_xgb_mgrv')

# Native pixel size of the base-learner rasters (about 4.77 m). Computed once.
NATIVE_SCALE = knn.projection().nominalScale().getInfo()
print(f'Base-learner native scale: {NATIVE_SCALE:.3f} m')

## 4. Per-point statistics (interpreters over a 10 m buffer; models at the centroid pixel)

In [ ]:
def compute_point_stats(feat):
    geom = feat.geometry()
    buf = geom.buffer(10)
    # Interpreter SD aggregates the three interpreters within the 10 m buffer.
    # The buffer matches the same physical point across the three interpreter assets.
    i1 = ee.Number(fc1.filterBounds(buf).aggregate_mean('val_int'))
    i2 = ee.Number(fc2.filterBounds(buf).aggregate_mean('val_int'))
    i3 = ee.Number(fc3.filterBounds(buf).aggregate_mean('val_int'))
    interp_mean = i1.add(i2).add(i3).divide(3)
    interp_mean_sq = i1.pow(2).add(i2.pow(2)).add(i3.pow(2)).divide(3)
    interp_std = interp_mean_sq.subtract(interp_mean.pow(2)).max(0).sqrt()
    # Base-learner probabilities sampled at the centroid pixel (native ~4.77 m).
    knn_v    = sample_image_at_point(knn,    geom, NATIVE_SCALE)
    logreg_v = sample_image_at_point(logreg, geom, NATIVE_SCALE)
    rf_v     = sample_image_at_point(rf,     geom, NATIVE_SCALE)
    svc_v    = sample_image_at_point(svc,    geom, NATIVE_SCALE)
    xgb_v    = sample_image_at_point(xgb,    geom, NATIVE_SCALE)
    models_mean = knn_v.add(logreg_v).add(rf_v).add(svc_v).add(xgb_v).divide(5)
    models_mean_sq = knn_v.pow(2).add(logreg_v.pow(2)).add(rf_v.pow(2)).add(svc_v.pow(2)).add(xgb_v.pow(2)).divide(5)
    models_std = models_mean_sq.subtract(models_mean.pow(2)).max(0).sqrt()
    return feat.set({
        'i1_mean': i1, 'i2_mean': i2, 'i3_mean': i3,
        'interp_std': interp_std,
        'knn_prob': knn_v, 'logreg_prob': logreg_v, 'rf_prob': rf_v,
        'svc_prob': svc_v, 'xgb_prob': xgb_v,
        'models_std': models_std
    })

result_fc = fc1.map(compute_point_stats)
df = ee_fc_to_df(result_fc)
print(f'Rows pulled from GEE: {len(df)}')
df.head()

## 5. Clean DataFrame, preserve stratum, save to CSV

In [ ]:
cols = ['source', 'i1_mean', 'i2_mean', 'i3_mean', 'interp_std',
        'knn_prob', 'logreg_prob', 'rf_prob', 'svc_prob', 'xgb_prob', 'models_std']
df = df[cols].dropna()
df['stratum'] = df['source'].map({'cert': 'Consensus', 'conf': 'Disagreement'})
df.to_csv(OUT_DIR / 'reference_points_with_stats.csv', index=False)
print(f'Total points: {len(df)}')
print(df['stratum'].value_counts())

## 6. Spearman correlations (overall and per-stratum)

In [ ]:
def spearman(x, y):
    rho, p = stats.spearmanr(x.astype(float), y.astype(float))
    return rho, p

rho_all, p_all = spearman(df['interp_std'], df['models_std'])
rho_c,   p_c   = spearman(df.loc[df['source']=='cert', 'interp_std'], df.loc[df['source']=='cert', 'models_std'])
rho_d,   p_d   = spearman(df.loc[df['source']=='conf', 'interp_std'], df.loc[df['source']=='conf', 'models_std'])

summary = [
    f'Spearman rho (overall, n={len(df)}): rho = {rho_all:.4f}, p = {p_all:.4g}',
    f'Spearman rho (consensus, n={(df["source"]=="cert").sum()}): rho = {rho_c:.4f}, p = {p_c:.4g}',
    f'Spearman rho (disagreement, n={(df["source"]=="conf").sum()}): rho = {rho_d:.4f}, p = {p_d:.4g}',
]
for line in summary:
    print(line)
(OUT_DIR / 'spearman_correlation.txt').write_text('\n'.join(summary), encoding='utf-8')

## 7. Two-panel figure

In [ ]:
def cdf_xy(values):
    v = np.sort(np.asarray(values, dtype=float))
    y = np.arange(1, len(v) + 1) / len(v)
    return v, y

fig, (ax_cdf, ax_sc) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [1, 1.2]})

model_x, model_y = cdf_xy(df['models_std'])
interp_x, interp_y = cdf_xy(df['interp_std'])
ax_cdf.plot(model_x,  model_y,  color='black', linestyle='-',  linewidth=2,
            label='Model SD (across 5 base learners)')
ax_cdf.plot(interp_x, interp_y, color='black', linestyle='--', linewidth=2,
            label='Interpreter SD (across 3 interpreters)')
ax_cdf.set_xlabel('Standard deviation')
ax_cdf.set_ylabel('Cumulative fraction of pixels')
ax_cdf.set_xlim(-0.02, 0.55)
ax_cdf.set_ylim(0, 1.02)
ax_cdf.grid(True, linestyle='--', alpha=0.4)
# Legend with white background covers the CDF lines beneath
ax_cdf.legend(loc='lower right', frameon=True, framealpha=0.95,
              facecolor='white', edgecolor='gray', fontsize=9)
ax_cdf.set_title('A. Distribution of variability', loc='left', fontsize=11, fontweight='bold')

m_c = (df['source'] == 'cert')
m_d = (df['source'] == 'conf')
ax_sc.scatter(df.loc[m_c, 'interp_std'].astype(float), df.loc[m_c, 'models_std'].astype(float),
              c=COLOR_CONSENSUS, alpha=0.7, edgecolor='black', linewidth=0.4, s=40,
              label=f'Consensus stratum (n={m_c.sum()})')
ax_sc.scatter(df.loc[m_d, 'interp_std'].astype(float), df.loc[m_d, 'models_std'].astype(float),
              c=COLOR_DISAGREE,  alpha=0.7, edgecolor='black', linewidth=0.4, s=40,
              label=f'Disagreement stratum (n={m_d.sum()})')

lim = max(0.55, float(df['interp_std'].astype(float).max()), float(df['models_std'].astype(float).max()))
ax_sc.plot([0, lim], [0, lim], '--', color='gray', linewidth=1, label='1:1 reference')

ax_sc.set_xlabel('Interpreter SD (across 3 interpreters)')
ax_sc.set_ylabel('Model SD (across 5 base learners)')
ax_sc.set_xlim(-0.02, lim)
ax_sc.set_ylim(-0.02, lim)
ax_sc.grid(True, linestyle='--', alpha=0.4)
ax_sc.set_aspect('equal', adjustable='box')
ax_sc.set_title('B. Joint distribution', loc='left', fontsize=11, fontweight='bold')

# Spearman annotation
ax_sc.text(0.027, 0.70,
           f'Spearman $\\rho$ = {rho_all:.2f}\n(p = {p_all:.2g}, n = {len(df)})',
           transform=ax_sc.transAxes,
           fontsize=10, verticalalignment='bottom',
           bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.85))
# Legend
ax_sc.legend(loc='upper left', frameon=True, framealpha=0.95,
             facecolor='white', edgecolor='gray', fontsize=9)

plt.tight_layout()
fig.savefig(OUT_DIR / 'figure5_redesigned.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'figure5_redesigned.pdf', bbox_inches='tight')
plt.show()
print(f'Figure saved to {OUT_DIR}')